In [5]:
!pip install faiss-cpu pypdf
import os
import re
import textwrap
from pathlib import Path

import faiss
import numpy as np
import requests
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [4]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 28.1 MB/s eta 0:00:00


In [15]:
# ---------------------------------------------------------------------------
# 1. Config
# ---------------------------------------------------------------------------
PDF_URL = "https://www.berkshirehathaway.com/letters/2024ltr.pdf"
PDF_PATH = Path("./2024ltr.pdf")

EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # 33M params, ~120MB, strong on MTEB
CHUNK_SIZE = 800                          # chars (roughly ~200 tokens)
CHUNK_OVERLAP = 150                       # 15-20% overlap is a safe default
TOP_K = 4                                 # retrieved chunks per query

# OpenAI-compatible endpoint. Swap base_url + model to point elsewhere.
LLM_CLIENT = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", "INSERT_YOUR_KEY_HERE"),
    base_url="https://api.openai.com/v1
    # base_url="https://api.together.xyz/v1",         # Together
    # base_url="http://localhost:11434/v1",           # Ollama
)
LLM_MODEL = "gpt-4o-mini"                 # or "meta-llama/Llama-3.1-8B-Instruct-Turbo" on Together, etc.

In [7]:
# ---------------------------------------------------------------------------
# 2. Download PDF (idempotent)
# ---------------------------------------------------------------------------
if not PDF_PATH.exists():
    print(f"Downloading {PDF_URL} ...")
    r = requests.get(PDF_URL, timeout=60)
    r.raise_for_status()
    PDF_PATH.write_bytes(r.content)
print(f"PDF: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.0f} KB)")

PDF: 2024ltr.pdf (63 KB)


In [8]:
# ---------------------------------------------------------------------------
# 3. Extract text page-by-page
# ---------------------------------------------------------------------------
# Keeping page numbers as metadata is essential for grounded answers — without
# them, the model can cite a chunk but the user can't verify.
reader = PdfReader(str(PDF_PATH))
pages = []
for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    # pypdf often leaves hard linebreaks mid-sentence; fold them.
    text = re.sub(r"-\n", "", text)            # de-hyphenate line-broken words
    text = re.sub(r"\n{2,}", "\n\n", text)     # preserve paragraph breaks
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)  # join soft wraps
    text = re.sub(r"[ \t]+", " ", text).strip()
    if text:
        pages.append({"page": i, "text": text})
print(f"Extracted {len(pages)} non-empty pages")

Extracted 15 non-empty pages


In [9]:
# ---------------------------------------------------------------------------
# 4. Chunk with overlap
# ---------------------------------------------------------------------------
def chunk_text(text: str, size: int, overlap: int):
    """Simple char-based sliding window. For production, switch to a token-aware
    splitter (e.g. tiktoken-based) or a semantic splitter. This is good enough
    to demonstrate retrieval mechanics without extra dependencies."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        # Try not to cut mid-sentence: extend to the next period within 100 chars
        if end < len(text):
            next_period = text.find(". ", end - 50, end + 100)
            if next_period != -1:
                end = next_period + 1
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [c for c in chunks if c]

records = []  # list of {"id", "page", "text"}
for p in pages:
    for j, chunk in enumerate(chunk_text(p["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
        records.append({
            "id": f"p{p['page']}-c{j}",
            "page": p["page"],
            "text": chunk,
        })
print(f"Built {len(records)} chunks "
      f"(avg {sum(len(r['text']) for r in records) // len(records)} chars)")

Built 60 chunks (avg 728 chars)


In [10]:
# ---------------------------------------------------------------------------
# 5. Embed
# ---------------------------------------------------------------------------
# bge-small-en-v1.5 recommends prefixing queries (not passages) with a short
# instruction; that asymmetric prompting noticeably improves retrieval.
embedder = SentenceTransformer(EMBED_MODEL)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

print("Embedding chunks ...")
chunk_embeddings = embedder.encode(
    [r["text"] for r in records],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # so inner product == cosine similarity
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
# ---------------------------------------------------------------------------
# 6. FAISS index (cosine = inner product on L2-normalized vectors)
# ---------------------------------------------------------------------------
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")


FAISS index: 60 vectors, dim=384


In [12]:
# ---------------------------------------------------------------------------
# 7. Retrieve
# ---------------------------------------------------------------------------
def retrieve(query: str, k: int = TOP_K):
    q_emb = embedder.encode(
        [QUERY_PREFIX + query], normalize_embeddings=True
    ).astype("float32")
    scores, idxs = index.search(q_emb, k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        rec = records[idx]
        hits.append({**rec, "score": float(score)})
    return hits

In [13]:
# ---------------------------------------------------------------------------
# 8. Generate (RAG prompt)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = textwrap.dedent("""
    You are a financial analyst assistant answering questions about
    Berkshire Hathaway's 2024 shareholder letter.

    Rules:
    - Answer ONLY from the provided context. If the context does not contain
      the answer, say "The letter does not address this directly."
    - Cite the page number(s) you used in square brackets, e.g. [p. 5].
    - Be concise. Quote figures exactly as they appear in the source.
""").strip()

def build_user_prompt(question: str, hits: list[dict]) -> str:
    context_blocks = "\n\n".join(
        f"[Chunk {h['id']} | p. {h['page']} | score={h['score']:.3f}]\n{h['text']}"
        for h in hits
    )
    return f"Context:\n{context_blocks}\n\nQuestion: {question}"

def rag_answer(question: str, k: int = TOP_K, show_context: bool = False) -> str:
    hits = retrieve(question, k=k)
    user_prompt = build_user_prompt(question, hits)

    if show_context:
        print("\n--- Retrieved chunks ---")
        for h in hits:
            preview = h["text"][:160].replace("\n", " ")
            print(f"  p.{h['page']:>3} score={h['score']:.3f}  {preview}...")
        print()

    resp = LLM_CLIENT.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,    # factual QA — keep it deterministic
    )
    return resp.choices[0].message.content

In [16]:
# ---------------------------------------------------------------------------
# 9. Demo queries
# ---------------------------------------------------------------------------
demo_questions = [
    "How much did Berkshire pay in U.S. federal income taxes in 2024?",
    "What did Buffett say about share repurchases this year?",
    "Which Japanese trading companies does Berkshire hold, and what is the plan for them?",
    "What does Buffett say about Greg Abel as his successor?",
    "What was Berkshire's operating earnings in 2024?",
]

for q in demo_questions:
    print("=" * 78)
    print(f"Q: {q}")
    print("-" * 78)
    print(rag_answer(q, show_context=True))
    print()

Q: How much did Berkshire pay in U.S. federal income taxes in 2024?
------------------------------------------------------------------------------

--- Retrieved chunks ---
  p.  5 score=0.757  income tax than the U.S. government had ever received from any company – even the American tech titans that commanded market values in the trillions. To be prec...
  p.  5 score=0.718  7 $37,350 * Includes certain businesses in which Berkshire had between a 20% and 50% ownership such as Kraft Heinz, Occidental Petroleum and Berkadia. ** Includ...
  p.  6 score=0.686  can be hard to visualize. Let me recast the $26.8 billion that we paid last year. If Berkshire had sent the Treasury a $1 million check every 20 minutes through...
  p.  5 score=0.683  Here’s a breakdown of the 2023-24 earnings as we see them. All calculations are after depreciation, amortization and income tax. EBITDA, a flawed favorite of Wa...

Berkshire paid a total of $26.8 billion in U.S. federal income taxes in 2024 [p. 5].



In [19]:
# ---------------------------------------------------------------------------
# 9. Demo queries
# ---------------------------------------------------------------------------
demo_questions = [
    "create an analysis on breakdown of the 2023-24 earnings",

]

for q in demo_questions:
    print("=" * 78)
    print(f"Q: {q}")
    print("-" * 78)
    print(rag_answer(q, show_context=True))
    print()

Q: create an analysis on breakdown of the 2023-24 earnings
------------------------------------------------------------------------------

--- Retrieved chunks ---
  p.  5 score=0.763  Here’s a breakdown of the 2023-24 earnings as we see them. All calculations are after depreciation, amortization and income tax. EBITDA, a flawed favorite of Wa...
  p.  4 score=0.701  ity operation from about 92% to 100% at a cost of roughly $3.9 billion, of which $2.9 billion was paid in cash with a balance in Berkshire “B” shares. * * * * *...
  p.  6 score=0.688  can be hard to visualize. Let me recast the $26.8 billion that we paid last year. If Berkshire had sent the Treasury a $1 million check every 20 minutes through...
  p. 11 score=0.661  fixed rates, no “floaters.” Greg and I have no view on future foreign exchange rates and therefore seek a position approximating currency-neutrality. We are req...

The breakdown of Berkshire Hathaway's 2023-24 earnings shows a significant increase in operatin